In [1]:
from langchain_openai.chat_models import ChatOpenAI
from pydantic import BaseModel
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate

In [2]:
load_dotenv()

False

In [3]:
text = """Artificial intelligence is transforming technology and shaping the future.
Machine learning algorithms are becoming more sophisticated every day.
Deep learning models can now process vast amounts of data efficiently.
Neural networks are inspired by the human brain's structure.
The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.
Climate change is affecting ecosystems worldwide.
Rising temperatures are causing glaciers to melt at unprecedented rates.
Scientists warn that immediate action is needed to reduce carbon emissions.
Renewable energy sources offer hope for a sustainable future."""

In [4]:
# pydantic class for structured output

class Chunk(BaseModel): 
    
    chunk_text: str
    summary: str
    
    
class Chunker(BaseModel):
    
    chunks: list[Chunk]

In [13]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.output_parsers import PydanticOutputParser

class Chunk(BaseModel):
    text: str = Field(description="Original text of the chunk without modifications")
    summary: str = Field(description="1-2 line summary of the chunk")

class Chunker(BaseModel):
    chunks: List[Chunk]
    
# define model
from langchain_ollama import ChatOllama
# model = ChatOpenAI(model="gpt-5-mini")
# Define Ollama model
model = ChatOllama(
    model="qwen3.5:9b",
    temperature=0
)

parser = PydanticOutputParser(pydantic_object=Chunker)



# llm_chunker = model.with_structured_output(schema=Chunker)

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate(
    messages=[
        ("system", """
        You are an expert Text Chunker.

        Your task is to divide the given text into semantically meaningful
        chunks based on natural topic boundaries.

        Follow these rules:

        1. Split the text whenever there is a meaningful change in topic,
           subject, or context.
        2. Preserve the original text exactly as provided.
        3. Do not rewrite, paraphrase, correct, or remove any information.
        4. Avoid creating chunks that are too small or too large.
        5. Each chunk should represent a coherent topic or idea.
        6. Generate a concise 1-2 line summary for each chunk.
        7. Ensure that all original text is covered without duplication.

        Return ONLY valid JSON in the following format:

        {{
            "chunks": [
                {{
                    "text": "Original text of the first chunk",
                    "summary": "Summary of the first chunk"
                }},
                {{
                    "text": "Original text of the second chunk",
                    "summary": "Summary of the second chunk"
                }}
            ]
        }}
        """),
        ("human", "Split the given text into chunks:\n\n{text}")
    ],
    input_variables=["text"]
)

In [15]:
# chunking through llm

model_chain = prompt | model | parser

response = model_chain.invoke({"text": text})

In [16]:
response

Chunker(chunks=[Chunk(text="Artificial intelligence is transforming technology and shaping the future.\nMachine learning algorithms are becoming more sophisticated every day.\nDeep learning models can now process vast amounts of data efficiently.\nNeural networks are inspired by the human brain's structure.", summary='Describes the evolution and impact of artificial intelligence, machine learning, and neural networks on technology.'), Chunk(text='The best pasta recipes include fresh ingredients and proper cooking techniques.\nItalian cuisine emphasizes quality olive oil and regional cheeses.\nAuthentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.\nCooking pasta al dente ensures the best texture and flavor.', summary='Details Italian culinary practices, specific ingredients for carbonara, and proper pasta cooking techniques.'), Chunk(text='Climate change is affecting ecosystems worldwide.\nRising temperatures are causing glaciers to melt at unprecedented rates.\nSc

In [39]:
response.chunks

[Chunk(chunk_text="Artificial intelligence is transforming technology and shaping the future.\nMachine learning algorithms are becoming more sophisticated every day.\nDeep learning models can now process vast amounts of data efficiently.\nNeural networks are inspired by the human brain's structure.", summary="Overview of artificial intelligence topics: AI's impact, machine learning, deep learning, and neural networks."),
 Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques.\nItalian cuisine emphasizes quality olive oil and regional cheeses.\nAuthentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.\nCooking pasta al dente ensures the best texture and flavor.', summary='Notes on pasta and Italian cuisine: ingredients, authentic carbonara, and cooking pasta al dente.'),
 Chunk(chunk_text='Climate change is affecting ecosystems worldwide.\nRising temperatures are causing glaciers to melt at unprecedented rates.\nScientists wa

In [17]:
len(response.chunks)

3

In [22]:
chunks = response.chunks
chunks


[Chunk(text="Artificial intelligence is transforming technology and shaping the future.\nMachine learning algorithms are becoming more sophisticated every day.\nDeep learning models can now process vast amounts of data efficiently.\nNeural networks are inspired by the human brain's structure.", summary='Describes the evolution and impact of artificial intelligence, machine learning, and neural networks on technology.'),
 Chunk(text='The best pasta recipes include fresh ingredients and proper cooking techniques.\nItalian cuisine emphasizes quality olive oil and regional cheeses.\nAuthentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.\nCooking pasta al dente ensures the best texture and flavor.', summary='Details Italian culinary practices, specific ingredients for carbonara, and proper pasta cooking techniques.'),
 Chunk(text='Climate change is affecting ecosystems worldwide.\nRising temperatures are causing glaciers to melt at unprecedented rates.\nScientists warn

In [19]:
from termcolor import COLORS, colored
from random import choice

In [20]:
def display_chunks(chunks):
    colors_list = list(COLORS.keys())[2:8]
    print(f"Total Number of Chunks: {len(chunks)}")
    
    for num, chunk in enumerate(chunks, 1):
        print(f"Chunk {num}: Length {len(chunk)} chars")
        print(colored(text=chunk, color=choice(colors_list)), end="\n\n")

In [23]:
display_chunks([chunk.text for chunk in chunks])

Total Number of Chunks: 3
Chunk 1: Length 277 chars
Artificial intelligence is transforming technology and shaping the future.
Machine learning algorithms are becoming more sophisticated every day.
Deep learning models can now process vast amounts of data efficiently.
Neural networks are inspired by the human brain's structure.

Chunk 2: Length 283 chars
The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.

Chunk 3: Length 260 chars
Climate change is affecting ecosystems worldwide.
Rising temperatures are causing glaciers to melt at unprecedented rates.
Scientists warn that immediate action is needed to reduce carbon emissions.
Renewable energy sources offer hope for a sustainable future.



In [24]:
response.chunks[2]

Chunk(text='Climate change is affecting ecosystems worldwide.\nRising temperatures are causing glaciers to melt at unprecedented rates.\nScientists warn that immediate action is needed to reduce carbon emissions.\nRenewable energy sources offer hope for a sustainable future.', summary='Addresses the environmental impact of climate change and highlights renewable energy as a solution.')

In [25]:
# create documents from chunks

docs = [Document(page_content=chunk.text, metadata={"summary": chunk.summary}) for chunk in chunks]

In [26]:
print(docs)

[Document(metadata={'summary': 'Describes the evolution and impact of artificial intelligence, machine learning, and neural networks on technology.'}, page_content="Artificial intelligence is transforming technology and shaping the future.\nMachine learning algorithms are becoming more sophisticated every day.\nDeep learning models can now process vast amounts of data efficiently.\nNeural networks are inspired by the human brain's structure."), Document(metadata={'summary': 'Details Italian culinary practices, specific ingredients for carbonara, and proper pasta cooking techniques.'}, page_content='The best pasta recipes include fresh ingredients and proper cooking techniques.\nItalian cuisine emphasizes quality olive oil and regional cheeses.\nAuthentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.\nCooking pasta al dente ensures the best texture and flavor.'), Document(metadata={'summary': 'Addresses the environmental impact of climate change and highlights renew

In [27]:
print(docs[1])

page_content='The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.' metadata={'summary': 'Details Italian culinary practices, specific ingredients for carbonara, and proper pasta cooking techniques.'}
